# Bronze Layer ETL Pipeline

**Purpose:** Load raw CSV data into bronze layer tables in the DataWarehouse catalog.

**Prerequisites:**
1. Upload CSV files to cloud storage (DBFS, Unity Catalog Volumes, or S3)
2. Update file paths in the configuration section below
3. Ensure the DataWarehouse catalog and bronze schema exist
4. Grant necessary permissions to read from source paths and write to bronze tables

**Tables Loaded:**
- `bronze.crm_cust_info`
- `bronze.crm_prd_info`
- `bronze.crm_sales_details`
- `bronze.erp_cust_az12`
- `bronze.erp_loc_a101`
- `bronze.erp_px_cat_g1v1`

**Process:** Truncate and load pattern with error handling and execution timing.

In [0]:
import time
from datetime import datetime
from pyspark.sql import SparkSession

# Configuration
CATALOG_NAME = "DataWarehouse"
SCHEMA_NAME = "bronze"

# TODO: Update these paths after uploading CSV files to cloud storage
# Options:
# - DBFS: /dbfs/FileStore/bronze_data/
# - Volumes: /Volumes/DataWarehouse/bronze/raw_data/
# - S3: s3://your-bucket/bronze_data/
BASE_PATH = "/Volumes/datawarehouse/bronze/csv_data_files/datasets"  

# File mappings: table_name -> (csv_filename, display_name)
TABLE_CONFIG = {
    # CRM Tables
    "crm_cust_info": ("cust_info.csv", "CRM Customer Info"),
    "crm_prd_info": ("prd_info.csv", "CRM Product Info"),
    "crm_sales_details": ("sales_details.csv", "CRM Sales Details"),
    # ERP Tables
    "erp_cust_az12": ("CUST_AZ12.csv", "ERP Customer AZ12"),
    "erp_loc_a101": ("LOC_A101.csv", "ERP Location A101"),
    "erp_px_cat_g1v1": ("PX_CAT_G1V2.csv", "ERP Product Category G1V1"),
}

print("✓ Configuration loaded")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {SCHEMA_NAME}")
print(f"  Base Path: {BASE_PATH}")
print(f"  Tables to load: {len(TABLE_CONFIG)}")

✓ Configuration loaded
  Catalog: DataWarehouse
  Schema: bronze
  Base Path: /Volumes/datawarehouse/bronze/csv_data_files/datasets
  Tables to load: 6


In [0]:
%sql
USE CATALOG DataWarehouse;

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType, DateType, IntegerType, DoubleType

def load_table_from_csv(table_name, csv_filename, display_name):
    """
    Load data from CSV into a bronze table using truncate-and-load pattern.
    
    Args:
        table_name: Name of the table in bronze schema (without schema prefix)
        csv_filename: Name of the CSV file
        display_name: Human-readable name for logging
    
    Returns:
        Tuple of (success: bool, duration: float, error_message: str)
    """
    start_time = time.time()
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    file_path = f"{BASE_PATH}/{csv_filename}"
    
    try:
        # Print status: Truncating
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Truncating table {full_table_name}...")
        spark.sql(f"TRUNCATE TABLE {full_table_name}")
        
        # Print status: Loading
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Loading data from {csv_filename}...")
        
        # Get target table schema
        target_schema = spark.table(full_table_name).schema
        
        # Read CSV with all columns as strings first (to avoid inference issues)
        df = spark.read.csv(
            file_path,
            header=True,
            inferSchema=False  # Read everything as string initially
        )
        
        # Normalize column names to lowercase (handles case sensitivity issues)
        df = df.toDF(*[c.lower() for c in df.columns])
        
        # Cast columns to match target table schema
        for field in target_schema.fields:
            if field.name in df.columns:
                df = df.withColumn(field.name, col(field.name).cast(field.dataType))
        
        # Write to Delta table
        df.write.format("delta").mode("append").saveAsTable(full_table_name)
        
        rows_loaded = df.count()
        duration = time.time() - start_time
        print(f"✓ Loaded {rows_loaded} rows into {display_name} in {duration:.2f} seconds")
        
        return (True, duration, None)
        
    except Exception as e:
        duration = time.time() - start_time
        error_msg = f"✗ ERROR loading {display_name}: {str(e)}"
        print(error_msg)
        print(f"  Table: {full_table_name}")
        print(f"  File: {file_path}")
        print(f"  Duration: {duration:.2f} seconds")
        
        return (False, duration, str(e))


def print_separator():
    """Print a separator line between table loads."""
    print("-" * 80)


print("✓ Helper functions defined")

✓ Helper functions defined


In [0]:
print("=" * 80)
print("LOADING CRM TABLES")
print("=" * 80)

crm_start_time = time.time()
crm_results = []

# Load CRM Customer Info
result = load_table_from_csv("crm_cust_info", "cust_info.csv", "CRM Customer Info")
crm_results.append(("crm_cust_info", result))
print_separator()

# Load CRM Product Info
result = load_table_from_csv("crm_prd_info", "prd_info.csv", "CRM Product Info")
crm_results.append(("crm_prd_info", result))
print_separator()

# Load CRM Sales Details
result = load_table_from_csv("crm_sales_details", "sales_details.csv", "CRM Sales Details")
crm_results.append(("crm_sales_details", result))
print_separator()

crm_duration = time.time() - crm_start_time
crm_success_count = sum(1 for _, (success, _, _) in crm_results if success)

print(f"\n✓ CRM Tables Complete: {crm_success_count}/{len(crm_results)} successful in {crm_duration:.2f} seconds\n")

LOADING CRM TABLES
[2026-05-06 07:30:35] Truncating table DataWarehouse.bronze.crm_cust_info...
[2026-05-06 07:30:37] Loading data from cust_info.csv...
✓ Loaded 18494 rows into CRM Customer Info in 4.66 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:30:40] Truncating table DataWarehouse.bronze.crm_prd_info...
[2026-05-06 07:30:41] Loading data from prd_info.csv...
✓ Loaded 397 rows into CRM Product Info in 4.03 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:30:44] Truncating table DataWarehouse.bronze.crm_sales_details...
[2026-05-06 07:30:45] Loading data from sales_details.csv...
✓ Loaded 60398 rows into CRM Sales Details in 4.27 seconds
--------------------------------------------------------------------------------

✓ CRM Tables Complete: 3/3 successful in 12.96 seconds



In [0]:
print("=" * 80)
print("LOADING ERP TABLES")
print("=" * 80)

erp_start_time = time.time()
erp_results = []

# Load ERP Customer AZ12
result = load_table_from_csv("erp_cust_az12", "CUST_AZ12.csv", "ERP Customer AZ12")
erp_results.append(("erp_cust_az12", result))
print_separator()

# Load ERP Location A101
result = load_table_from_csv("erp_loc_a101", "LOC_A101.csv", "ERP Location A101")
erp_results.append(("erp_loc_a101", result))
print_separator()

# Load ERP Product Category G1V1
result = load_table_from_csv("erp_px_cat_g1v1", "PX_CAT_G1V2.csv", "ERP Product Category G1V1")
erp_results.append(("erp_px_cat_g1v1", result))
print_separator()

erp_duration = time.time() - erp_start_time
erp_success_count = sum(1 for _, (success, _, _) in erp_results if success)

print(f"\n✓ ERP Tables Complete: {erp_success_count}/{len(erp_results)} successful in {erp_duration:.2f} seconds\n")

LOADING ERP TABLES
[2026-05-06 07:33:09] Truncating table DataWarehouse.bronze.erp_cust_az12...
[2026-05-06 07:33:10] Loading data from CUST_AZ12.csv...
✓ Loaded 18484 rows into ERP Customer AZ12 in 4.50 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:33:14] Truncating table DataWarehouse.bronze.erp_loc_a101...
[2026-05-06 07:33:15] Loading data from LOC_A101.csv...
✓ Loaded 18484 rows into ERP Location A101 in 3.87 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:33:18] Truncating table DataWarehouse.bronze.erp_px_cat_g1v1...
[2026-05-06 07:33:19] Loading data from PX_CAT_G1V2.csv...
✓ Loaded 37 rows into ERP Product Category G1V1 in 3.92 seconds
--------------------------------------------------------------------------------

✓ ERP Tables Complete: 3/3 successful in 12.29 seconds



In [0]:
print("=" * 80)
print("BRONZE LAYER LOAD SUMMARY")
print("=" * 80)

# Combine all results
all_results = crm_results + erp_results
total_duration = crm_duration + erp_duration
total_success = crm_success_count + erp_success_count
total_tables = len(all_results)

print(f"\nExecution Time: {total_duration:.2f} seconds")
print(f"Tables Processed: {total_tables}")
print(f"Successful: {total_success}")
print(f"Failed: {total_tables - total_success}")

# Detailed results
print("\nDetailed Results:")
for table_name, (success, duration, error) in all_results:
    status = "✓ SUCCESS" if success else "✗ FAILED"
    print(f"  {status} | {table_name:<25} | {duration:>6.2f}s")
    if error:
        print(f"           Error: {error}")

# Check for failures
if total_success < total_tables:
    print("\n⚠ WARNING: Some tables failed to load. Review errors above.")
    failed_tables = [name for name, (success, _, _) in all_results if not success]
    print(f"Failed tables: {', '.join(failed_tables)}")
else:
    print("\n✓ All bronze tables loaded successfully!")

print("=" * 80)

BRONZE LAYER LOAD SUMMARY

Execution Time: 25.25 seconds
Tables Processed: 6
Successful: 6
Failed: 0

Detailed Results:
  ✓ SUCCESS | crm_cust_info             |   4.66s
  ✓ SUCCESS | crm_prd_info              |   4.03s
  ✓ SUCCESS | crm_sales_details         |   4.27s
  ✓ SUCCESS | erp_cust_az12             |   4.50s
  ✓ SUCCESS | erp_loc_a101              |   3.87s
  ✓ SUCCESS | erp_px_cat_g1v1           |   3.92s

✓ All bronze tables loaded successfully!
